In [2]:
pip install mlxtend

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.4.3-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.4.3-cp311-cp311-win_amd64.whl (12.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.31.0 requires numpy<2,>=1.19.3, but you have numpy 2.4.3 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import mlxtend
import streamlit
import numpy
print(numpy.__version__)

1.26.4


In [4]:
import pandas as pd

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [5]:
df = pd.read_csv(r"C:\Users\asmis\OneDrive\Desktop\foodbridge\data\food_master_dataset.csv")

df.head()

,date,city,district,outlet_type,total_prepared_kg,total_sold_kg,surplus_kg,footfall_count,rainfall_mm,temperature_c,...,unemployment_rate,malnutrition_rate,literacy_rate,population_density,rainfall_category,temperature_category,surplus_category,feedback_source,feedback_text,sentiment_label
0,2024-04-12,Chennai,Mumbai Suburban,restaurant,234.27,178.91,55.36,936,33.819924,22.980879,...,9.281988,23.345097,71.159051,13857.50378,medium,cool,medium,logistics_team,Food supply was slightly different than expect...,neutral
1,2025-03-11,Chennai,Hyderabad,restaurant,216.43,169.38,47.05,611,57.927890,27.431975,...,5.543214,31.585074,57.331012,27642.76583,medium,moderate,medium,restaurant_staff,Coordination with volunteers worked great and ...,positive
2,2024-09-27,Chennai,Chennai,restaurant,222.05,135.79,86.26,1004,19.304584,25.671839,...,2.607615,37.821608,74.279447,22704.52683,low,moderate,medium,restaurant_staff,Food supply was slightly different than expect...,neutral
3,2024-04-16,Hyderabad,Chennai,restaurant,351.51,329.97,21.54,787,10.248639,27.042423,...,5.059368,15.320457,74.441768,19956.61007,low,moderate,low,logistics_team,Community support was excellent and food reach...,positive
4,2024-03-12,Bangalore,Hyderabad,supermarket,406.52,355.98,50.54,805,23.895929,24.664987,...,13.216180,8.737902,85.915223,14806.84788,medium,cool,medium,volunteer,Shelter operations were routine with no major ...,neutral


In [6]:
df["surplus_level"] = pd.cut(
    df["surplus_kg"],
    bins=[0,20,60,1000],
    labels=["low_surplus","medium_surplus","high_surplus"]
)

df["prep_level"] = pd.cut(
    df["total_prepared_kg"],
    bins=[0,100,300,1000],
    labels=["low_prep","medium_prep","high_prep"]
)

df["footfall_level"] = pd.cut(
    df["footfall_count"],
    bins=[0,80,200,1000],
    labels=["low_footfall","medium_footfall","high_footfall"]
)

In [7]:
transactions = df[
    [
        "surplus_level",
        "prep_level",
        "footfall_level",
        "rainfall_category",
        "temperature_category",
        "outlet_type"
    ]
]

In [8]:
transactions_encoded = pd.get_dummies(transactions)

transactions_encoded.head()

,surplus_level_low_surplus,surplus_level_medium_surplus,surplus_level_high_surplus,prep_level_low_prep,prep_level_medium_prep,prep_level_high_prep,footfall_level_low_footfall,footfall_level_medium_footfall,footfall_level_high_footfall,rainfall_category_high,rainfall_category_low,rainfall_category_medium,temperature_category_cool,temperature_category_hot,temperature_category_moderate,outlet_type_farm,outlet_type_restaurant,outlet_type_supermarket
0,False,True,False,False,True,False,False,False,True,False,False,True,True,False,False,False,True,False
1,False,True,False,False,True,False,False,False,True,False,False,True,False,False,True,False,True,False
2,False,False,True,False,True,False,False,False,False,False,True,False,False,False,True,False,True,False
3,False,True,False,False,False,True,False,False,True,False,True,False,False,False,True,False,True,False
4,False,True,False,False,False,True,False,False,True,False,False,True,True,False,False,False,False,True


In [9]:
frequent_itemsets = apriori(
    transactions_encoded,
    min_support=0.05,
    use_colnames=True
)

frequent_itemsets.sort_values("support", ascending=False).head()

,support,itemsets
5,0.8392,(footfall_level_high_footfall)
12,0.6025,(outlet_type_restaurant)
6,0.5968,(rainfall_category_low)
4,0.5437,(prep_level_high_prep)
2,0.5271,(surplus_level_high_surplus)


In [10]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.6
)

rules[["antecedents","consequents","support","confidence","lift"]].head(10)

,antecedents,consequents,support,confidence,lift
0,(surplus_level_low_surplus),(prep_level_medium_prep),0.0669,0.916438,2.011056
1,(surplus_level_low_surplus),(footfall_level_high_footfall),0.0619,0.847945,1.010421
2,(surplus_level_low_surplus),(outlet_type_restaurant),0.0646,0.884932,1.468766
3,(surplus_level_medium_surplus),(prep_level_medium_prep),0.2498,0.624656,1.370762
4,(surplus_level_medium_surplus),(footfall_level_high_footfall),0.3364,0.841210,1.002395
5,(surplus_level_medium_surplus),(rainfall_category_low),0.2404,0.601150,1.007289
6,(surplus_level_medium_surplus),(outlet_type_restaurant),0.2841,0.710428,1.179133
7,(prep_level_high_prep),(surplus_level_high_surplus),0.3875,0.712709,1.352133
8,(surplus_level_high_surplus),(prep_level_high_prep),0.3875,0.735155,1.352133
9,(surplus_level_high_surplus),(footfall_level_high_footfall),0.4409,0.836464,0.996739


In [11]:
rules.sort_values(by="lift", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
184,(surplus_level_low_surplus),"(outlet_type_restaurant, footfall_level_high_f...",0.0730,0.3129,0.0519,0.710959,2.272160,1.0,0.029058,2.377171,0.603981,0.155389,0.579332,0.438413
181,"(surplus_level_low_surplus, footfall_level_hig...","(outlet_type_restaurant, prep_level_medium_prep)",0.0619,0.3704,0.0519,0.838449,2.263632,1.0,0.028972,3.897224,0.595067,0.136435,0.743407,0.489284
37,(surplus_level_low_surplus),"(outlet_type_restaurant, prep_level_medium_prep)",0.0730,0.3704,0.0609,0.834247,2.252286,1.0,0.033861,3.798413,0.599791,0.159216,0.736732,0.499332
265,(outlet_type_farm),"(prep_level_high_prep, surplus_level_high_surp...",0.0986,0.3255,0.0686,0.695740,2.137451,1.0,0.036506,2.216857,0.590363,0.192968,0.548911,0.453247
264,"(footfall_level_high_footfall, outlet_type_farm)","(prep_level_high_prep, surplus_level_high_surp...",0.0836,0.3875,0.0686,0.820574,2.117611,1.0,0.036205,3.413667,0.575916,0.170435,0.707060,0.498803
87,(outlet_type_farm),"(prep_level_high_prep, surplus_level_high_surp...",0.0986,0.3875,0.0801,0.812373,2.096447,1.0,0.041893,3.264459,0.580211,0.197291,0.693671,0.509541
182,"(surplus_level_low_surplus, outlet_type_restau...","(footfall_level_high_footfall, prep_level_medi...",0.0646,0.3841,0.0519,0.803406,2.091657,1.0,0.027087,3.132846,0.557954,0.130796,0.680801,0.469263
178,"(surplus_level_low_surplus, footfall_level_hig...",(prep_level_medium_prep),0.0549,0.4557,0.0519,0.945355,2.074512,1.0,0.026882,9.960690,0.548047,0.113146,0.899605,0.529623
36,"(surplus_level_low_surplus, outlet_type_restau...",(prep_level_medium_prep),0.0646,0.4557,0.0609,0.942724,2.068739,1.0,0.031462,9.503184,0.552292,0.132564,0.894772,0.538183
34,(surplus_level_low_surplus),"(footfall_level_high_footfall, prep_level_medi...",0.0730,0.3841,0.0567,0.776712,2.022162,1.0,0.028661,2.758325,0.545286,0.141608,0.637461,0.462165
